## **1. Setup**
---

In [1]:
import os
import time
import pickle
import warnings

import cv2
import faiss
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

import timm
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image
from facenet_pytorch import MTCNN

warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
DEVICE    = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
IMG_SIZE  = 112
EMB_DIM   = 128
THRESHOLD = 0.7
print(f'Using device: {DEVICE}')

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

MODEL_PATH = 'results/models/best_model.pt'
INDEX_PATH = 'gallery/embedding/face_index.index'
INFO_PATH  = 'gallery/user/face_info.pkl'
os.makedirs(os.path.dirname(INDEX_PATH), exist_ok=True)
os.makedirs(os.path.dirname(INFO_PATH), exist_ok=True)

Using device: cuda:0


## **2. Embedding Model**
---

In [3]:
class EmbeddingModel(nn.Module):
    def __init__(self, backbone='efficientnet_b0', n_layers_unfreeze=10):
        super().__init__()
        
        self.backbone = timm.create_model(backbone, pretrained=True)
        self.backbone.classifier = nn.Identity()

        for p in self.backbone.parameters():                            p.requires_grad = False
        for p in list(self.backbone.parameters())[-n_layers_unfreeze:]: p.requires_grad = True

        feat_dim = self.backbone.num_features
        self.embedding_head = nn.Sequential(
            nn.Linear(feat_dim, 512), nn.SiLU(), nn.BatchNorm1d(512), nn.Dropout(0.2),
            nn.Linear(512, 256)     , nn.SiLU(), nn.BatchNorm1d(256), nn.Dropout(0.1),
            nn.Linear(256, 128)     , nn.SiLU(), nn.BatchNorm1d(128),
            nn.Linear(128, 128)
        )

    def forward(self, x):
        feat = self.backbone(x)
        emb  = self.embedding_head(feat)
        emb  = F.normalize(emb, p=2, dim=1)
        return emb
    
def load_model(model, model_path, device):
    state = torch.load(model_path, map_location=device)

    model.backbone.load_state_dict(state['model_backbone'])
    model.embedding_head.load_state_dict(state['model_embedding'])
    model.to(device)
    return model.eval()

In [4]:
emb_model     = load_model(EmbeddingModel(), MODEL_PATH, DEVICE)
face_detector = MTCNN(image_size=IMG_SIZE, margin=10, keep_all=False, post_process=False, device=DEVICE)

In [5]:
def get_embedding(emb_model, img_input):
    if isinstance(img_input, str):
        img = Image.open(img_input).convert('RGB')
        
    elif isinstance(img_input, Image.Image):
        img = img_input.convert('RGB')
        
    elif isinstance(img_input, np.ndarray):
        img = Image.fromarray(cv2.cvtColor(img_input, cv2.COLOR_BGR2RGB))
        
    elif isinstance(img_input, torch.Tensor):
        # img = img_input.permute(1, 2, 0).cpu().numpy()
        # img = Image.fromarray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        img = to_pil_image(img_input.cpu())
        
    else:
        raise TypeError(f"Unsupported input type: {type(img_input)}")
        
        
    img = transform(img).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        emb = emb_model(img).cpu().numpy()
    return emb.reshape(1, -1).astype(np.float32)

## **3. Index & Info**
---

In [6]:
def build_index_and_info(emb_dim, index_path, info_path):
    if os.path.exists(index_path):
        index = faiss.read_index(index_path)
    else:
        index = faiss.IndexFlatIP(emb_dim)
        faiss.write_index(index, index_path)
        
    if os.path.exists(info_path):
        info = pickle.load(open(info_path,'rb'))
    else:
        info = []
        pickle.dump(info, open(info_path,'wb'))

    return index, info

In [7]:
def get_index_and_info(index_path, info_path):
    index = faiss.read_index(index_path)
    info  = pickle.load(open(info_path,'rb'))
    return index, info

## **4. Function**
---

In [8]:
def detect_face():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print('Error: Could not open webcam.')
        return

    prev_time = 0
    
    while True:
        ret, frame = cap.read()
        frame = cv2.flip(frame, 1)
        if not ret:
            print('Error: Could not capture frame.')
            break
            
        # Calculate FPS
        current_time = time.time()
        fps = 1 / (current_time - prev_time) if prev_time > 0 else 0
        prev_time = current_time
        cv2.putText(frame, f"FPS: {fps:.2f}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        # Detect face
        bbox, prob = face_detector.detect(frame)
        draw_frame = frame.copy()
        
        if bbox is not None:
            for i, (box, confidence) in enumerate(zip(bbox, prob)):
                x1, y1, x2, y2 = [int(v) for v in box]
                cv2.rectangle(draw_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(draw_frame, f"{confidence:.2f}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
                  
        cv2.imshow('Face Detection ([ESC] to escape) (Test camera)', draw_frame)
        if cv2.waitKey(30) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()

In [9]:
def add_face():
    index, info = get_index_and_info(INDEX_PATH, INFO_PATH)
    pid         = input('Enter person ID: ')
    name        = input('Enter person name: ')
    
    if pid in [i['pid'] for i in info]:
        print(f"Person ID {pid} already exists. Please use a different ID.")
        return
    
    else:
        face_cnt  = 0
        face_embs = []
        pdir      = os.path.join('gallery/user', pid)
        os.makedirs(pdir, exist_ok=True)
        
        
        cap = cv2.VideoCapture(0)
        while face_cnt < 10:
            ret, frame = cap.read()
            frame      = cv2.flip(frame, 1)
            if not ret: break
            
            bbox, _  = face_detector.detect(frame)
            if bbox is not None and len(bbox) == 1:
                x1, y1, x2, y2 = [int(v) for v in bbox[0]]
                draw_frame     = frame.copy()
                cv2.rectangle(draw_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                pil_img   = Image.fromarray(rgb_frame) 
                face_img  = face_detector.extract(pil_img, bbox, save_path=os.path.join(pdir, f"{os.path.basename(pdir)}_{face_cnt:03d}.jpg"))
                face_cnt += 1
                
                if face_img is not None:
                    print(f"Extracted face {face_cnt} from {pid}.")
                    face_embs.append(get_embedding(emb_model, face_img))
            
            time.sleep(0.05)
            
            cv2.imshow('Face Registration', draw_frame)
            if cv2.waitKey(30) & 0xFF == 27:
                break

        cap.release()
        cv2.destroyAllWindows()

        
        for face_emb in face_embs:
            info.append({'pid': pid, 'name': name, 'embedding': face_emb})
        pickle.dump(info, open(INFO_PATH, 'wb'))
        
        face_embs = np.concatenate(face_embs, axis=0)
        index.add(face_embs)
        faiss.write_index(index, INDEX_PATH)
        
        print(f"Added person ID {pid} to index.")
        return info, index

In [10]:
def remove_face():
    index, info = get_index_and_info(INDEX_PATH, INFO_PATH)
    pid         = input('Enter person ID to remove: ')
    
    if pid not in [i['pid'] for i in info]:
        print(f"Person ID {pid} does not exist.")
        return
    
    else:
        # Delete the person folder and images
        pdir = os.path.join('gallery/user', pid)
        if os.path.exists(pdir):
            for file in os.listdir(pdir):
                os.remove(os.path.join(pdir, file))
            os.rmdir(pdir)
        
        info  = [i for i in info if i['pid'] != pid]
        index = faiss.IndexFlatIP(128)
        for i in info:
            index.add(i['embedding'].reshape(1, -1))
        faiss.write_index(index, INDEX_PATH)
        pickle.dump(info, open(INFO_PATH, 'wb'))
        
        print(f"Removed person ID {pid} from index.")
        return info, index

In [11]:
def search_face():
    index, info = get_index_and_info(INDEX_PATH, INFO_PATH)
    cap         = cv2.VideoCapture(0)
    if index.ntotal == 0:
        print("No faces registered in the system.")
        return
    
    while True:
        ret, frame = cap.read()
        frame      = cv2.flip(frame, 1)
        if not ret: break

        bbox, _    = face_detector.detect(frame)
        draw_frame = frame.copy()
        if bbox is not None and len(bbox) == 1:
            x1, y1, x2, y2 = [int(v) for v in bbox[0]]
            cv2.rectangle(draw_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            
            rgb_frame  = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img    = Image.fromarray(rgb_frame) 
            face_img   = face_detector.extract(pil_img, bbox, save_path=None)
            
            face_emb   = get_embedding(emb_model, face_img)
            D, I       = index.search(face_emb, 1)
            score, idx = D[0][0], I[0][0]
            
            if score >= THRESHOLD:
                name = info[idx]['name']
                cv2.putText(draw_frame, f"{name} ({score:.2f})", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
            else:
                cv2.putText(draw_frame, 'Unknown', (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

        time.sleep(0.05)
        cv2.imshow('Face Recognition', draw_frame)
        if cv2.waitKey(30) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()

## **5. Testing**
---

In [12]:
index, info = build_index_and_info(128, INDEX_PATH, INFO_PATH)
print(f'Index size: {index.ntotal}')
print(f'Info size: {len(info)}')

Index size: 20
Info size: 20


In [13]:
# Test camera
detect_face()

In [105]:
# Add face
index, info = add_face()

Extracted face 1 from 2.
Extracted face 2 from 2.
Extracted face 3 from 2.
Extracted face 4 from 2.
Extracted face 5 from 2.
Extracted face 6 from 2.
Extracted face 7 from 2.
Extracted face 8 from 2.
Extracted face 9 from 2.
Extracted face 10 from 2.
Added person ID 2 to index.


In [99]:
index, info = build_index_and_info(128, INDEX_PATH, INFO_PATH)
print(f'Index size: {index.ntotal}')
print(f'Info size: {len(info)}')

Index size: 20
Info size: 20


In [ ]:
# Remove face
index, info = remove_face()

Removed person ID 2 from index.


([{'pid': '0',
   'name': 'Tai',
   'embedding': array([[ 0.19260156, -0.08230321,  0.06935804,  0.01014982,  0.01924257,
            0.05195054,  0.10127631, -0.12363631, -0.13521867, -0.0962173 ,
            0.23984896,  0.12841311, -0.02763944,  0.01248288, -0.11430948,
            0.01069726,  0.10510845,  0.03064556, -0.08640284, -0.09363025,
           -0.22058725, -0.01964518,  0.02808158, -0.0686819 , -0.07201848,
           -0.04682309,  0.08067831, -0.06090378, -0.05515089,  0.2297022 ,
           -0.06442162, -0.09241616, -0.09771203,  0.13787787, -0.01773074,
           -0.05670042, -0.02988644, -0.02650042,  0.08471379,  0.04008354,
           -0.24766311, -0.11973096, -0.06208549, -0.05230062,  0.0278505 ,
           -0.10753176, -0.06022659,  0.00741465,  0.10276591, -0.01627551,
           -0.01532273,  0.0593469 ,  0.03422905, -0.08854523, -0.06318413,
            0.00876499,  0.04685311, -0.01567134,  0.21430261, -0.02601505,
            0.08945418,  0.10056105, -0.05

In [14]:
index, info = get_index_and_info(INDEX_PATH, INFO_PATH)
print(f'Index size: {index.ntotal}')
print(f'Info size: {len(info)}')

Index size: 20
Info size: 20


In [15]:
# show all user info
for i in info:
    print(f"ID: {i['pid']}, Name: {i['name']}, Embedding: {i['embedding'].shape}")

ID: 0, Name: Tai, Embedding: (1, 128)
ID: 0, Name: Tai, Embedding: (1, 128)
ID: 0, Name: Tai, Embedding: (1, 128)
ID: 0, Name: Tai, Embedding: (1, 128)
ID: 0, Name: Tai, Embedding: (1, 128)
ID: 0, Name: Tai, Embedding: (1, 128)
ID: 0, Name: Tai, Embedding: (1, 128)
ID: 0, Name: Tai, Embedding: (1, 128)
ID: 0, Name: Tai, Embedding: (1, 128)
ID: 0, Name: Tai, Embedding: (1, 128)
ID: 1, Name: BA, Embedding: (1, 128)
ID: 1, Name: BA, Embedding: (1, 128)
ID: 1, Name: BA, Embedding: (1, 128)
ID: 1, Name: BA, Embedding: (1, 128)
ID: 1, Name: BA, Embedding: (1, 128)
ID: 1, Name: BA, Embedding: (1, 128)
ID: 1, Name: BA, Embedding: (1, 128)
ID: 1, Name: BA, Embedding: (1, 128)
ID: 1, Name: BA, Embedding: (1, 128)
ID: 1, Name: BA, Embedding: (1, 128)


In [16]:
# Search face
search_face()